# Baseline Model (V0)

## Load and clean data

In [2]:
import numpy as np
import pandas as pd

In [3]:
df_train = pd.read_parquet('../data/train.parquet')
df_test = pd.read_parquet('../data/test.parquet')
df_sub = pd.read_csv('../data/example_submission.csv')

We use the same code that we use in 01 to get df_train_clean, df_test_clean and y_train:

In [4]:
# y_train
churn_users = df_train.loc[df_train['page'] == 'Cancellation Confirmation', 'userId'].unique()
all_users = pd.Index(sorted(df_train['userId'].unique()), name='userId')
y_train = pd.Series(all_users.isin(churn_users).astype(int), index= all_users, name='target')

# df_train_clean
df_train_clean = df_train[
    (df_train['auth']=='Logged In') & (~df_train['page'].isin(['Cancellation Confirmation', 'Cancel']))
    ]
# df_test_clean
df_test_clean = df_test[df_test['auth']=='Logged In']

# we delete the other datasets
del df_train, df_test

## Feature Engineering (V0)

We write a single `build_features(df)` function so `train` and `test` get the exact same transformations.  
For this V0, we keep it minimal (19 numeric features):
- `pd.crosstab(..., normalize='index')` gives the proportion of each `page` per user (independent of how long the user stayed, so we avoid the `max_time` / raw count bias from the EDA).
- `pct_paid`: share of events while on a paid plan, and `is_male`: binary encoding of `gender`.
- `.reindex(...)` makes sure `X_train` matches the exact order of `y_train`, and `X_test` matches `df_sub['id']` (including the anonymous user `1261737`, filled with 0).

In [8]:
def build_features(df):
    features = pd.crosstab(df['userId'], df['page'], normalize='index')
    features['pct_paid'] = (df['level']=='paid').groupby(df['userId']).mean()
    features['is_male'] = (df['gender']=='M').groupby(df['userId']).first().astype(int)
    return features

In [9]:
X_train = build_features(df_train_clean).reindex(y_train.index)
X_test = build_features(df_test_clean).reindex(df_sub['id'].astype(str)).fillna(0)

display(X_train.head())

page,About,Add Friend,Add to Playlist,Downgrade,Error,Help,Home,Logout,NextSong,Roll Advert,Save Settings,Settings,Submit Downgrade,Submit Upgrade,Thumbs Down,Thumbs Up,Upgrade,pct_paid,is_male
userId,,,,,,,,,,,,,,,,,,,
1000025,0.000499,0.014978,0.026460,0.008987,0.000499,0.003994,0.038442,0.012981,0.829755,0.003495,0.001498,0.003994,0.0,0.000499,0.006490,0.046930,0.000499,0.973040,1
1000035,0.001285,0.014781,0.017352,0.004499,0.000643,0.003213,0.034704,0.012211,0.813625,0.003856,0.000643,0.004499,0.0,0.000643,0.009640,0.075193,0.003213,0.897172,0
1000083,0.000000,0.010101,0.013468,0.003367,0.000000,0.003367,0.035354,0.023569,0.843434,0.013468,0.000000,0.008418,0.0,0.001684,0.003367,0.035354,0.005051,0.624579,1
1000103,0.000000,0.000000,0.013333,0.013333,0.000000,0.000000,0.080000,0.026667,0.760000,0.040000,0.000000,0.000000,0.0,0.013333,0.013333,0.026667,0.013333,0.400000,0
1000164,0.001946,0.016537,0.023346,0.009728,0.000973,0.003891,0.038911,0.010700,0.823930,0.019455,0.000973,0.004864,0.0,0.000973,0.005837,0.036965,0.000973,0.783074,0


## Baseline Model & Cross-Validation

We use `HistGradientBoostingClassifier` for the baseline because it is fast, works very well on tabular data (as seen in Lab 3), and does not require feature scaling.
- `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`: keeps the exact 22.3% churn ratio in every fold, and shuffles the users since `userId` is sorted.
- `class_weight='balanced'`: penalizes mistakes on churners ~3.5x more during training because they are the minority class.
- `scoring='balanced_accuracy'`: evaluates with the exact Kaggle metric instead of regular accuracy.

In [25]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5,shuffle=True, random_state=42)
model = HistGradientBoostingClassifier(random_state=42, class_weight='balanced')

scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='balanced_accuracy')

print(f"The cross val scores are {np.round(scores, 4)}")
print(f"Mean CV Balanced Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")


The cross val scores are [0.6522 0.6465 0.6625 0.6794 0.6715]
Mean CV Balanced Accuracy: 0.6624 (+/- 0.0121)


## First Submission (V0).

`cross_val_score` only evaluates on folds and does not keep `model` fitted, so we now fit `model` on 100% of `X_train` and predict on `X_test`.  
We force user `1261737` to `0` (it never logged in), run basic sanity checks (shape, ID order, NaNs, predicted churn ratio), and export with `index=False` so Kaggle only gets the `id,target` columns.

In [52]:
model.fit(X_train, y_train)
preds = model.predict(X_test)
sub = pd.DataFrame(
    {'id':df_sub['id'], 'target':preds})
sub.loc[sub['id']==1261737, 'target']=0

print("Shape:", sub.shape)
print("IDs Aligned ?", (sub['id'] == df_sub['id']).all())
print("Missing values:", sub.isna().sum().sum())
sub.to_csv('../../submissions/sub_v0_baseline.csv', index=False) # otherwise it exports 3 columns...
print(sub['target'].value_counts(normalize=True))

Shape: (2904, 2)
IDs Aligned ? True
Missing values: 0
target
0    0.606061
1    0.393939
Name: proportion, dtype: float64


## Adversarial Validation (V0)

To check if any of our 19 features behaves differently between train and test, we stack `X_train` (label 0) and `X_test` (label 1) and train a classifier to separate them using ROC-AUC.  
An AUC close to 0.5 means train and test look alike (good); an AUC > 0.8 would signal a leak or distribution shift.

In [70]:
X_adv = pd.concat([X_train, X_test], axis=0)

y_adv = np.concatenate([np.zeros(len(X_train)), np.ones(len(X_test))])

adv_model = HistGradientBoostingClassifier(random_state=42)

adv_scores = cross_val_score(adv_model, X_adv, y_adv, cv=cv, scoring= 'roc_auc')
print(f"Adversarial Validation AUC: {adv_scores.mean():.4f} (+/- {adv_scores.std():.4f})")

Adversarial Validation AUC: 0.5891 (+/- 0.0152)


### V0 Baseline Summary:
- Simple features (17 `page` proportions + `pct_paid` + `is_male`) with `HistGradientBoostingClassifier(class_weight='balanced')`.
- 5-fold Stratified CV Balanced Accuracy: **0.6624 (+/- 0.0121)**.
- First Kaggle Public Leaderboard score (`sub_v0_baseline.csv`): **0.61363** (benchmark is 0.52547).
- Adversarial Validation AUC: **0.5891** -> very close to 0.5, meaning our normalized features do not suffer from a major train/test distribution shift.
- Note: on `X_test`, the model currently predicts 39.4% churners (vs 22.3% in train) -> tuning the decision threshold later should help.